**EDA Fase 1: Diagnóstico de RECH0 (Características del Hogar / Geografía)**

Objetivo: Descubrir empíricamente qué variables y etiquetas contiene realmente este módulo en la ventana histórica 2007-2024. 

No haremos suposiciones de filtrado hasta que imprimamos la distribución de nulos y sus diccionarios de variables.

In [ ]:
import pandas as pd
import gc
from mnp.ingestion.loader import load_endes

# 1. Carga de historial desde 2007 para RECH0
history = load_endes(year=range(2007, 2025), module='household', record='household_characteristics', meta=True)

frames = []
diccionario = {}

# Extraemos y liberamos memoria progresivamente usando pop()
for yr in sorted(list(history.keys())):
    df_yr, meta_yr = history.pop(yr)
    frames.append(df_yr.assign(year=yr))
    diccionario.update(meta_yr.column_names_to_labels)

df_rech0 = pd.concat(frames, ignore_index=True)
del frames
gc.collect() # Forzar limpieza de memoria

print(f"Registros RECH0 totales: {len(df_rech0)}")
print(f"Columnas RECH0 totales: {df_rech0.shape[1]}")

In [ ]:
# 2. Descubrimiento: Porcentaje de nulos y etiquetas de la data real
nulos = df_rech0.isna().mean().sort_values(ascending=False) * 100

print("Variables con > 90% de nulos:")
for col, pct in nulos[nulos > 90].items():
    etiqueta = diccionario.get(col, 'Sin etiqueta')
    print(f"{col:7s} | {pct:5.1f}% | {etiqueta}")

print("\nResto de variables:")
for col, pct in nulos[nulos <= 90].items():
    etiqueta = diccionario.get(col, 'Sin etiqueta')
    print(f"{col:7s} | {pct:5.1f}% | {etiqueta}")